In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import numpy as np

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Katalog roboczy (notatnik): {notebook_dir}")
print(f"Katalog główny projektu: {project_root}")
print(f"Załadowano ścieżkę modułów: {src_path}")

In [ ]:
from hep_tracking.models import (
    knn_numpy_brute_force,
    knn_cupy_brute_force,
    knn_scipy_ckdtree,
    knn_sklearn_kdtree,
    knn_sklearn_balltree,
)
from hep_tracking.plots import plot_3d_hits, plot_distance_distributions
from hep_tracking.utils import measure_execution_time

print("Zależności modułu hep_tracking załadowane pomyślnie!")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

data_dir = project_root / "data"
sanity_dataset = data_dir / "dataset_hard_10k.npz"

if sanity_dataset.exists():
    loaded_data = np.load(sanity_dataset)
    features_sanity = loaded_data["X"]
    labels_sanity = loaded_data["y"]
    
    print(f"Wczytano zbiór kontrolny: {sanity_dataset.name}")
    print(f"Kształt macierzy cech: {features_sanity.shape}")
    
    plot_3d_hits(features_sanity, labels_sanity)
    plt.show()
    
    plot_distance_distributions(features_sanity, labels_sanity)
    plt.show()
else:
    print(f"Nie znaleziono pliku {sanity_dataset.name}. Upewnij się, że wygenerowałeś dane (make generate).")

In [ ]:
target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
dataset_modes = ["easy", "hard"]
k_neighbors = 5

methods = {
    "Numpy Brute Force": knn_numpy_brute_force,
    "CuPy Brute Force": knn_cupy_brute_force,
    "Scipy cKDTree": knn_scipy_ckdtree,
    "Sklearn KDTree": knn_sklearn_kdtree,
    "Sklearn BallTree": knn_sklearn_balltree,
}

results = {mode: {method: [] for method in methods} for mode in dataset_modes}

In [ ]:
import warnings

warnings.filterwarnings('ignore')

for mode in dataset_modes:
    print(f"\n{'='*45}")
    print(f" TRYB DANYCH: {mode.upper()}")
    print(f"{'='*45}")

    for size_label, size_val in target_sizes.items():
        filename = data_dir / f"dataset_{mode}_{size_label}.npz"
        
        if not filename.exists():
            print(f"\n[BRAK PLIKU] {filename.name} - Pomijam.")
            for method_name in methods:
                results[mode][method_name].append(None)
            continue

        loaded_data = np.load(filename)
        features = loaded_data["X"]
        print(f"\n Załadowano: {size_label} ({features.shape[0]} hitów)")

        for method_name, method_callable in methods.items():
            
            if method_name in ["Numpy Brute Force", "CuPy Brute Force", "Sklearn BallTree"] and size_val > 100000:
                print(f"  > Pomijam: {method_name} (Zbyt wolne dla {size_label})")
                results[mode][method_name].append(None)
                continue

            def benchmark_wrapper():
                method_callable(features, k_neighbors)

            min_time = measure_execution_time(benchmark_wrapper, num_runs=3, warmup_runs=1)
            results[mode][method_name].append(min_time)
            
            print(f"  > {method_name}: {min_time * 1000:.2f} ms")

print("\nBenchmark zakończony sukcesem!")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
numeric_sizes = list(target_sizes.values())

for idx, mode in enumerate(dataset_modes):
    ax = axes[idx]
    
    for method_name, method_times in results[mode].items():
        valid_sizes = [s for s, t in zip(numeric_sizes, method_times) if t is not None]
        valid_times = [t for t in method_times if t is not None]
        
        if valid_times:
            ax.plot(
                valid_sizes,
                valid_times,
                marker="o",
                label=method_name,
                linewidth=2,
                markersize=6,
            )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(f"Skalowalność kNN - tryb: {mode.upper()}", fontsize=14)
    ax.set_xlabel("Liczba punktów (N)", fontsize=12)
    
    if idx == 0:
        ax.set_ylabel("Minimalny czas wykonania (s)", fontsize=12)
        
    ax.grid(True, which="both", ls="--", alpha=0.5)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()